In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Step 1: संरचित आऊटपुटसाठी पायडँटिक मॉडेल्स परिभाषित करा

हे मॉडेल्स एजंट परत करणाऱ्या **योजना** निर्धारित करतात. पायडँटिकसह `response_format` वापरल्याने सुनिश्चित होते:
- ✅ प्रकार-सुरक्षित डेटा निष्कर्षण
- ✅ स्वयंचलित प्रमाणीकरण
- ✅ मुक्त-टेक्स्ट प्रतिसादांमधून कोणतीही पार्सिंग त्रुटी नाहीत
- ✅ फील्ड्सवर आधारित सुलभ सशर्त राउटिंग


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: हॉटेल बुकिंग टूल तयार करा

हा टूलच **availability_agent** कॉल करेल की खोले उपलब्ध आहेत का ते तपासण्यासाठी. आम्ही `@ai_function` डेकोरेटर वापरतो जेणेकरून:
- Python फंक्शनला AI-कॉल करण्यायोग्य टूलमध्ये रूपांतर करता येईल
- LLM साठी JSON स्कीमा आपोआप तयार होते
- पॅरामीटरची पडताळणी हाताळता येईल
- एजंट्सद्वारे आपोआप कॉल होण्याची सोय असते

या डेमोमध्ये:
- **स्टॉकहोम, सिएटल, टोकियो, लंडन, ॲम्स्टर्डॅम** → खोले उपलब्ध आहेत ✅
- **इतर सर्व शहरं** → खोले नाहीत ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## टप्पा 3: राऊटिंगसाठी स्थिती फंक्शन्स परिभाषित करा

हे फंक्शन्स एजंटच्या प्रतिसादाची तपासणी करतात आणि कार्यप्रवाहात कोणता मार्ग घ्यायचा हे ठरवतात.

**महत्वाची पद्धत:**
1. संदेश `AgentExecutorResponse` आहे का तपासा
2. संरचित आउटपुट (Pydantic मॉडेल) पार्स करा
3. राऊटिंग नियंत्रित करण्यासाठी `True` किंवा `False` परत करा

कार्यप्रवाह हे अटी **एजेस** वर मूल्यांकन करेल ज्यामुळे पुढे कोणता executor वापरायचा ते ठरवले जाईल.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Step 4: सानुकूल प्रदर्शन कार्यान्वयक तयार करा

कार्यान्वयक workflow घटक असतात जे रूपांतरे किंवा साइड इफेक्ट्स करतात. आम्ही अंतिम निकाल दर्शविण्यासाठी `@executor` डेकॉरेटरचा वापर करून सानुकूल कार्यान्वयक तयार करतो.

**महत्वाचे संकल्पना:**
- `@executor(id="...")` - फंक्शनला workflow कार्यान्वयक म्हणून नोंदणी करतो
- `WorkflowContext[Never, str]` - इनपुट/आउटपुटसाठी टाइप हिन्ट्स
- `ctx.yield_output(...)` - अंतिम workflow निकाल परत करतो


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Step 5: पर्यावरण चल (Environment Variables) लोड करा

LLM ग्राहक (client) सेटअप करा. या उदाहरणात पुढील सोबत काम होते:
- **GitHub मॉडेल्स** (GitHub टोकनसह फ्री टियर)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Step 6: रचनेतून आउटपुटसह AI एजंट तयार करा

आम्ही **तीन विशेष एजंट** तयार करतो, प्रत्येक `AgentExecutor` मध्ये गुंडाळलेले:

1. **availability_agent** - साधन वापरून हॉटेलची उपलब्धता तपासतो
2. **alternative_agent** - पर्यायी शहरे सुचवतो (जेव्हा खोल्या उपलब्ध नसतील)
3. **booking_agent** - बुकिंगसाठी प्रोत्साहित करतो (जेव्हा खोल्या उपलब्ध असतील)

**महत्त्वाचे वैशिष्ट्ये:**
- `tools=[hotel_booking]` - एजंटला साधन प्रदान करते
- `response_format=PydanticModel` - रचनेतून JSON आउटपुट अनिवार्य करते
- `AgentExecutor(..., id="...")` - एजंटला वर्कफ्लोसाठी गुंडाळते


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## टप्पा ७: सशर्त काठांसह वर्कफ्लो तयार करा

आता आपण `WorkflowBuilder` वापरून सशर्त मार्गीकरणासह ग्राफ तयार करतो:

**वर्कफ्लो रचना:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**मुख्य पद्धती:**
- `.set_start_executor(...)` - प्रवेश बिंदू सेट करतो
- `.add_edge(from, to, condition=...)` - सशर्त काठ जोडतो
- `.build()` - वर्कफ्लो पूर्ण करतो


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: Test Case 1 चालवा - उपलब्धता नसलेले शहर (पॅरिस)

चला **कोणतीही उपलब्धता नाही** असा मार्ग तपासू या, पॅरिसमध्ये हॉटेल्सची विनंती करून (जिथे आमच्या सिम्युलेशनमध्ये रूमस नाहीत).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step ९: चाचणी प्रकरण २ चालवा - उपलब्धता असलेले शहर (स्टॉकहोम)

आता आपण स्टॉकहोममधील हॉटेल्ससाठी विनंती करून **उपलब्धता** मार्गाची चाचणी करूया (ज्याच्याकडे आमच्या सिम्युलेशनमध्ये खोल्या आहेत).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## मुख्य मुद्दे आणि पुढील पावले

### ✅ तुम्ही काय शिकलात:

1. **WorkflowBuilder पॅटर्न**
   - `.set_start_executor()` वापरून प्रवेश बिंदू निश्चित करा
   - `.add_edge(from, to, condition=...)` वापरून शर्तीगत मार्गदर्शन करा
   - वर्कफ्लो अंतिम करण्यासाठी `.build()` कॉल करा

2. **शर्तीगत मार्गदर्शन**
   - शर्ती फंक्शन्स `AgentExecutorResponse` तपासतात
   - मार्गदर्शन निर्णयांसाठी संरचित आउटपुट पार्स करा
   - एdge सक्रिय करण्यासाठी `True` आणि वगळण्यासाठी `False` परत करा

3. **टूल इंटिग्रेशन**
   - Python फंक्शन्सना AI टूल्समध्ये रूपांतरित करण्यासाठी `@ai_function` वापरा
   - आवश्यकतानुसार एजंट्स स्वयंचलितपणे टूल्स कॉल करतात
   - टूल्स JSON परत करतात, जे एजंट्स पार्स करू शकतात

4. **संरचित आउटपुट**
   - टाईप-सेफ डेटा एक्सट्रॅक्शनसाठी Pydantic मॉडेल्स वापरा
   - एजंट तयार करताना `response_format=MyModel` सेट करा
   - उत्तरं `Model.model_validate_json()` ने पार्स करा

5. **कस्टम Executor**
   - वर्कफ्लो कॉम्पोनंट तयार करण्यासाठी `@executor(id="...")` वापरा
   - Executor डेटा रूपांतरित करू शकतात किंवा साइड इफेक्ट्स करू शकतात
   - वर्कफ्लो परिणाम तयार करण्यासाठी `ctx.yield_output()` वापरा

### 🚀 वास्तविक-प्रकारच्या अनुप्रयोगांसाठी:

- **प्रवास बुकिंग**: उपलब्धता तपासा, पर्याय सुचवा, पर्यायांची तुलना करा
- **ग्राहक सेवा**: समस्या प्रकार, भावना, प्राधान्य यानुसार मार्गदर्शन करा
- **ई-कॉमर्स**: साठा तपासा, पर्याय सुचवा, ऑर्डर प्रक्रिया करा
- **सामग्री नियंत्रण**: विषारी गुणांकन, वापरकर्त्यांच्या फलकांनुसार मार्गदर्शन करा
- **मंजूरी वर्कफ्लो**: रक्कम, वापरकर्त्याची भूमिका, धोका पातळी यानुसार मार्गदर्शन करा
- **मल्टि-स्टेज प्रक्रिया**: डेटा गुणवत्ता, पूर्णत्वानुसार मार्गदर्शन करा

### 📚 पुढील पावले:

- अधिक गुंतागुंतीच्या शर्ती जोडा (अनेक निकष)
- वर्कफ्लो राज्य व्यवस्थापनासह लूप्स राबवा
- पुन्हा वापरता येण्याजोग्या घटकांसाठी उप-वर्कफ्लोज जोडा
- वास्तविक API सोबत इंटिग्रेट करा (हॉटेल बुकिंग, साठा प्रणाली)
- त्रुटी हाताळणी आणि फॉलबॅक मार्ग जोडा
- अंगभूत व्हिज्युअलायझेशन टूल्ससह वर्कफ्लोज दृष्टीकोन करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
